# Ensure Intacct Agent Service Principal

Idempotent setup task that:
1. Finds or creates a least-privilege Databricks service principal named `intacct-agent-{schema}`
2. Stores its `application_id` in the secret scope under `${client_id_dbs_key}`
3. Stores the workspace URL under `workspace_url`
4. Grants the SPN READ access to the secret scope
5. Outputs `application_id` as a task value for downstream DDL

The OAuth secret (under `${client_secret_dbs_key}`) is provisioned separately by an admin.

In [ ]:
dbutils.widgets.text('catalog_use', '')
dbutils.widgets.text('schema_use', '')
dbutils.widgets.text('secret_scope_name', '')
dbutils.widgets.text('client_id_dbs_key', '')
dbutils.widgets.text('client_secret_dbs_key', '')

catalog_use = dbutils.widgets.get('catalog_use')
schema_use = dbutils.widgets.get('schema_use')
secret_scope_name = dbutils.widgets.get('secret_scope_name')
client_id_dbs_key = dbutils.widgets.get('client_id_dbs_key')
client_secret_dbs_key = dbutils.widgets.get('client_secret_dbs_key')

spn_display_name = f'intacct-agent-{schema_use}'
print(f'SPN display name: {spn_display_name}')
print(f'Secret scope: {secret_scope_name}')
print(f'Client ID key: {client_id_dbs_key}')

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Locate or create the SPN
existing = [sp for sp in w.service_principals.list(filter=f"displayName eq '{spn_display_name}'")]
if existing:
    spn = existing[0]
    print(f'Found existing SPN: {spn.application_id}')
else:
    spn = w.service_principals.create(display_name=spn_display_name, active=True)
    print(f'Created new SPN: {spn.application_id}')

application_id = spn.application_id

In [ ]:
# Populate secret scope: client_id and workspace_url
workspace_url = f'https://{w.config.host}' if not w.config.host.startswith('http') else w.config.host

w.secrets.put_secret(scope=secret_scope_name, key=client_id_dbs_key, string_value=application_id)
w.secrets.put_secret(scope=secret_scope_name, key='workspace_url', string_value=workspace_url)
print(f'Stored {client_id_dbs_key} and workspace_url in scope {secret_scope_name}')

In [ ]:
# Grant the SPN READ access to its own scope
from databricks.sdk.service.workspace import AclPermission

try:
    w.secrets.put_acl(scope=secret_scope_name, principal=application_id, permission=AclPermission.READ)
    print(f'Granted READ on scope {secret_scope_name} to SPN {application_id}')
except Exception as e:
    print(f'put_acl raised: {e!r}; continuing (likely already granted)')

In [ ]:
# Output the application_id for the downstream DDL task
dbutils.jobs.taskValues.set(key='spn_application_id', value=application_id)
print(f'Set task value spn_application_id = {application_id}')